In [29]:
import os

# Find where you are right now
print("Current directory:", os.getcwd())

# List everything in current folder
print("\nFiles here:")
for item in os.listdir('.'):
    print(item)

Current directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/notebooks

Files here:
01_data_exploration.ipynb
02_workflow_1_direct_generation.ipynb
.ipynb_checkpoints


In [30]:
# Search for CSV files anywhere in your project
for root, dirs, files in os.walk('.'):
    # Skip venv folder
    if 'venv' in root:
        continue
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

In [31]:
import pandas as pd

# Update these paths to match where your files are
notes = pd.read_csv('../data/raw/clinical_notes.csv')
patients = pd.read_csv('../data/raw/patients.csv')
admissions = pd.read_csv('../data/raw/admissions.csv')

In [32]:
# Basic shape
print("=== NOTES ===")
print(notes.shape)
print(notes.columns.tolist())
print(notes.head(2))

print("\n=== PATIENTS ===")
print(patients.shape)
print(patients.columns.tolist())

print("\n=== ADMISSIONS ===")
print(admissions.shape)
print(admissions.columns.tolist())

=== NOTES ===
(1602, 9)
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
   ingest_timestamp                      clinical_note_id  \
0  07/01/2026 14:35  17bf845b-88f8-4604-8983-6e74453aada5   
1  07/01/2026 14:50  e5d9c0a4-299a-425e-abbc-27fabe9cb742   

                                     clean_note_text creation_timestamp  \
0  Patient Name: Judith Ada Wells\n- Patient ID: ...   07/01/2026 14:05   
1  Patient reviewed at 14:20 on 07/01/26 by Nurse...   07/01/2026 14:20   

         updt_dt_tm         note_subject note_type  \
0  07/01/2026 14:35            ED Triage        ED   
1  07/01/2026 14:50  ED Triage Follow-Up        ED   

                           admission_id                             person_id  
0  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  
1  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  

=== PATI

In [33]:
print("\n=== NOTE TYPES ===")
print(notes['note_type'].value_counts())


=== NOTE TYPES ===
note_type
Orthopaedics Inpatients                      263
Physiotherapy Documentation                  262
Medicine Inpatients                          242
ED                                           155
Neurology Inpatients                         104
Therapies Inpatients                          81
Paediatrics Inpatients                        66
Anaesthetic Documentation                     62
Respiratory Inpatients                        62
ED Depart Summary                             39
Respiratory Medicine Inpatients               37
Pre-op Checklist                              35
Pre-op Consent                                31
Theatre notes                                 31
Neurosurgery Inpatients                       26
Orthopaedics Outpatients                      24
Occupational Therapy Documentation            24
Surgery Inpatients                            21
Dietetics Documentation                       14
Speech and Language Therapy Documentati

In [34]:
# How many notes per admission?
notes_per_admission = notes.groupby('admission_id')['clinical_note_id'].count()
print("\n=== NOTES PER ADMISSION ===")
print(notes_per_admission.describe())
print(notes_per_admission.value_counts().head(10))

# How many admissions per patient?
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
print("\n=== ADMISSIONS PER PATIENT ===")
print(admissions_per_patient.describe())
print(admissions_per_patient.value_counts().head(10))


=== NOTES PER ADMISSION ===
count    69.000000
mean     23.217391
std       8.709139
min      10.000000
25%      16.000000
50%      22.000000
75%      30.000000
max      45.000000
Name: clinical_note_id, dtype: float64
clinical_note_id
22    5
25    5
33    4
17    4
23    4
19    4
35    4
15    4
12    3
26    3
Name: count, dtype: int64

=== ADMISSIONS PER PATIENT ===
count    50.000000
mean      1.380000
std       0.490314
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: admission_id, dtype: float64
admission_id
1    31
2    19
Name: count, dtype: int64


In [35]:
# Pick first patient and show all their notes in order
first_patient = notes['person_id'].iloc[0]
patient_notes = notes[notes['person_id'] == first_patient].sort_values('creation_timestamp')

print(f"\n=== FULL JOURNEY FOR PATIENT {first_patient} ===")
for _, row in patient_notes.iterrows():
    print(f"\nDate: {row['creation_timestamp']}")
    print(f"Type: {row['note_type']}")
    print(f"Text preview: {str(row['clean_note_text'])[:200]}")
    print("---")


=== FULL JOURNEY FOR PATIENT 28570119-9cdc-4120-98c0-4edb76cf36a3 ===

Date: 07/01/2026 14:05
Type: ED
Text preview: Patient Name: Judith Ada Wells
- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3
- NHS Number: 272733208
- Date of Birth: 15/05/84 (39 years old)
- Gender: Female
- Allergies: NKA
- Current Medicatio
---

Date: 07/01/2026 14:20
Type: ED
Text preview: Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurologic
---

Date: 07/01/2026 14:45
Type: ED
Text preview: - Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.
 - Date/Time: 07/01/26, 14:45.
 - Staff involved: Nurse Jasmine Freda Murray.
 - Chief Complaint: Severe headach
---

Date: 07/01/2026 15:15
Type: ED
Text preview: Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current 

In [36]:
# Check duration of admissions
notes['creation_timestamp'] = pd.to_datetime(
    notes['creation_timestamp'], 
    format='%d/%m/%Y %H:%M',
    dayfirst=True
)

date_range = notes.groupby('admission_id')['creation_timestamp'].agg(['min', 'max'])
date_range['duration_days'] = (date_range['max'] - date_range['min']).dt.days

print("=== ADMISSION DURATION ===")
print(date_range['duration_days'].describe())

# Check patients with 2 admissions
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
two_admission_patients = admissions_per_patient[admissions_per_patient == 2].index.tolist()
print(f"\nPatients with 2 admissions: {len(two_admission_patients)}")

# Check note types for multi-admission patients
multi_notes = notes[notes['person_id'].isin(two_admission_patients)]
print(f"Total notes for multi-admission patients: {len(multi_notes)}")

# Find discharge summary note type
print("\n=== DISCHARGE SUMMARY NOTES ===")
discharge_notes = notes[notes['note_type'].str.contains('Depart|Discharge|Summary', 
                                                          case=False, na=False)]
print(f"Count: {len(discharge_notes)}")
print(discharge_notes['note_type'].value_counts())

=== ADMISSION DURATION ===
count    69.000000
mean      8.869565
std       6.216470
min       0.000000
25%       4.000000
50%       7.000000
75%      15.000000
max      20.000000
Name: duration_days, dtype: float64

Patients with 2 admissions: 19
Total notes for multi-admission patients: 988

=== DISCHARGE SUMMARY NOTES ===
Count: 39
note_type
ED Depart Summary    39
Name: count, dtype: int64


In [37]:
# First confirm column names in both tables
print("Notes columns:", notes.columns.tolist())
print("Admissions columns:", admissions.columns.tolist())

# Check if person_id and patient_id contain same values
print("\nSample person_id from notes:", notes['person_id'].iloc[0])
print("Sample patient_id from admissions:", admissions['patient_id'].iloc[0])

Notes columns: ['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
Admissions columns: ['admission_id', 'admission_method', 'admission_timestamp', 'admission_title', 'bed_location', 'date_of_birth', 'nhs_number', 'patient_id', 'patient_name', 'site_id', 'site_name', 'ward', 'full_name', 'first_name', 'surname']

Sample person_id from notes: 28570119-9cdc-4120-98c0-4edb76cf36a3
Sample patient_id from admissions: 28570119-9cdc-4120-98c0-4edb76cf36a3


In [38]:
import pandas as pd

admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
two_admission_patients = admissions_per_patient[admissions_per_patient >= 2].index.tolist()
print(f"Patients with 2 admissions: {len(two_admission_patients)}")

Patients with 2 admissions: 19


In [39]:
multi_admission_with_gt = notes[
    (notes['person_id'].isin(two_admission_patients)) & 
    (notes['note_type'] == 'ED Depart Summary')
].groupby('person_id')['admission_id'].nunique()

both_admissions_have_gt = multi_admission_with_gt[
    multi_admission_with_gt == 2
].index.tolist()

print(f"Patients with ground truth for both admissions: {len(both_admissions_have_gt)}")

Patients with ground truth for both admissions: 10


In [40]:
# How many admissions do these 10 patients have total?
print(admissions[admissions['patient_id'].isin(both_admissions_have_gt)].groupby('patient_id')['admission_id'].count())

patient_id
04df53ea-55c1-48d9-84a1-1f15c133b29b    2
137b8481-4f1d-4b7f-babd-20f7117023ad    2
359014a1-10e6-4bd8-9ba7-513d021c971e    2
5e434d78-b2f6-4d88-b327-fff6ee50b901    2
61699d6d-904a-4ece-9026-cd61b3fd9a50    2
69bf7e25-abb2-4dde-857f-f1138d4d0d8a    2
8448b3fa-1f5e-45b5-bda5-eb0b7921b8cc    2
c50e236f-6b3d-41c8-9e16-7ec343cac820    2
c6c45c39-cd73-49dd-818d-0a7865fe8a7f    2
ff8c4724-b7de-4189-bccf-cffddd4d6d44    2
Name: admission_id, dtype: int64


In [41]:
import json

with open('../data/patients/longitudinal_patient_ids.json', 'w') as f:
    json.dump(both_admissions_have_gt, f)

print(f"Saved {len(both_admissions_have_gt)} patient IDs")

Saved 10 patient IDs


In [42]:
# Basic counts from the notes table
print("Number of notes:", len(notes))
print("Number of patients:", notes["person_id"].nunique())
print("Number of admissions:", notes["admission_id"].nunique())

# Check whether each patient has one or multiple admissions
admissions_per_patient = (
    notes.groupby("person_id")["admission_id"]
    .nunique()
)

print("\nAdmissions per patient:")
print(admissions_per_patient.describe())

print("\nPatients with more than one admission:")
print((admissions_per_patient > 1).sum())


# Calculate approximate note length in words
notes["word_count"] = (
    notes["clean_note_text"]
    .fillna("")
    .str.split()
    .str.len()
)

print("\nNote length statistics:")
print(
    notes["word_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Number of notes: 1602
Number of patients: 50
Number of admissions: 69

Admissions per patient:
count    50.000000
mean      1.380000
std       0.490314
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: admission_id, dtype: float64

Patients with more than one admission:
19

Note length statistics:
count    1602.000000
mean      129.997503
std        83.061291
min         1.000000
25%        60.250000
50%       124.500000
75%       174.000000
90%       228.900000
95%       280.000000
max       561.000000
Name: word_count, dtype: float64


In [43]:
notes_per_admission = (
    notes.groupby("admission_id")
    .size()
)

print("\nNotes per admission:")
print(
    notes_per_admission.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)


Notes per admission:
count    69.000000
mean     23.217391
std       8.709139
min      10.000000
25%      16.000000
50%      22.000000
75%      30.000000
90%      35.000000
95%      36.000000
max      45.000000
dtype: float64


In [44]:
longest_notes = notes.nlargest(10, "word_count")

for _, row in longest_notes.iterrows():
    print("=" * 80)
    print("Words:", row["word_count"])
    print("Subject:", row["note_subject"])
    print("Type:", row["note_type"])
    print()
    print(row["clean_note_text"])
    print()

Words: 561
Subject: Medical Clerking
Type: Neurology Inpatients

Clerking Doctor
Dr. Kelly Nicola Hayward (SpR)

Presenting Complaint
Acute confusion following minor fall

History of Presenting Complaint
- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)
- No LOC reported but recalls feeling dazed afterward
- Developed pprogressive confusion over the following hours
- Unable to remember recent events clearly
- Denies headache, visual changes, N&V, or limb weakness
- No reported CP, palpitations, or SOB

Review of Systems
- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures
- CVS: Denies chest pain, palpitations, or syncope
- Resp: No breathlessness or cough
- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit
- Renal: No dysuria or haematuria
- MSK: Reports mild right knee pain following the fall
- Endo: No polyuria, polydipsia, or heat/cold intolerance
- Other: No r

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

In [5]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
notes = pd.read_csv(DATA_DIR / "raw" / "clinical_notes.csv")
print("Original notes:", len(notes))
print(notes.columns.tolist())
print("Notes:", notes.shape)

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
Notes: (1602, 9)


In [8]:
# ============================================================
# SELECT PILOT PATIENTS BY LONGITUDINAL RECORD SIZE
#
# Restrict selection to the longitudinal cohort:
# patients with >= 2 admissions and ground truth for both.
# ============================================================

# ============================================================
# RECORD SIZE PER PATIENT
# ============================================================

patient_sizes = (
    notes
    .groupby("person_id")
    .size()
    .reset_index(name="note_count")
    .sort_values("note_count")
    .reset_index(drop=True)
)

display(patient_sizes)

print("\nNote-count distribution:")
print(patient_sizes["note_count"].describe())

,person_id,note_count
0,c87e610e-ac2a-48cf-ac32-054f3e595498,10
1,58b8aad6-7327-4450-956f-b775be0f4984,11
2,96766f3f-aa50-4bcb-abbf-884328049ea9,11
3,a8629d2e-7fd8-449a-9149-e0f5045a04c4,12
4,f97ee974-67f3-4d72-9ed4-f17025da3749,13
5,c54a9649-495a-4f89-a8ca-fbb14ba70ca6,14
6,900efb9a-fb57-4f25-b2a3-0a8c6f55b2d6,15
7,028998ee-babc-4096-9b28-001bc2f9a84e,15
8,0f438665-d430-4adb-8acc-c3beed9e4942,15
9,f1538231-21af-4cfa-8b40-3303aa0d0c96,15



Note-count distribution:
count    50.000000
mean     32.040000
std      19.973411
min      10.000000
25%      17.000000
50%      24.500000
75%      45.500000
max      90.000000
Name: note_count, dtype: float64


In [9]:
notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning + deduplication:", len(notes_dedup))

Raw notes: 1602
After cleaning + deduplication: 1103


In [10]:
patient_sizes_dedup = (
    notes_dedup
    .groupby("person_id")
    .size()
    .reset_index(name="note_count")
    .sort_values("note_count")
    .reset_index(drop=True)
)

display(patient_sizes_dedup)

print("\nDeduplicated note-count distribution:")
print(
    patient_sizes_dedup["note_count"].describe()
)

,person_id,note_count
0,ef943dfb-1368-44e0-a946-a0be360e1d54,10
1,c87e610e-ac2a-48cf-ac32-054f3e595498,10
2,96766f3f-aa50-4bcb-abbf-884328049ea9,11
3,58b8aad6-7327-4450-956f-b775be0f4984,11
4,f44d4a08-4c76-4000-ad61-d1d81032e643,12
5,a8629d2e-7fd8-449a-9149-e0f5045a04c4,12
6,f97ee974-67f3-4d72-9ed4-f17025da3749,13
7,1705dd0f-011a-492c-b006-b27e03f2f4ed,14
8,c54a9649-495a-4f89-a8ca-fbb14ba70ca6,14
9,028998ee-babc-4096-9b28-001bc2f9a84e,15



Deduplicated note-count distribution:
count    50.000000
mean     22.060000
std       8.471826
min      10.000000
25%      15.250000
50%      20.500000
75%      27.000000
max      45.000000
Name: note_count, dtype: float64


In [11]:
pilot_patients = {
    "short": "c87e610e-ac2a-48cf-ac32-054f3e595498",
    "medium": "ce0046dc-0ad3-4710-8147-549793c58b44",
    "long": "c6c45c39-cd73-49dd-818d-0a7865fe8a7f",
}

for case, patient_id in pilot_patients.items():
    n = len(
        notes_dedup[
            notes_dedup["person_id"] == patient_id
        ]
    )
    print(f"{case}: {patient_id} | {n} notes")

short: c87e610e-ac2a-48cf-ac32-054f3e595498 | 10 notes
medium: ce0046dc-0ad3-4710-8147-549793c58b44 | 21 notes
long: c6c45c39-cd73-49dd-818d-0a7865fe8a7f | 45 notes


In [12]:
# Short: 10 notes — c87e610e-ac2a-48cf-ac32-054f3e595498
# Medium: 21 notes — ce0046dc-0ad3-4710-8147-549793c58b44
# Long: 45 notes — c6c45c39-cd73-49dd-818d-0a7865fe8a7f